## BT4222 · MLP Feature Engineering (`mlp_feature_engineering.ipynb`)

**Feature groups produced** 

| Group | Features |
|---|---|
| Article | `article_content_emb` (MiniLM 384-d), `kg_entity_emb` (TransE 100-d) + `has_kg_entity`, `category_idx`, `subcategory_idx` |
| Popularity | `hist_ctr` (lagged, smoothed), `log_exposure` = log1p(lagged exposures), `is_cold_article` |
| User | `user_history_emb` (pooled MiniLM, 384-d), `log_history_length`, `has_history`, `reading_entropy`, `user_category_affinity` (for the shared category table) |
| User × Article | `category_match`, `cos_user_hist`, `max_cos_hist`, `cos_last5`, `cat_affinity`, `kg_match` |
| Context | `hour_sin`, `hour_cos` |
| Optional (ablation) | `diversity_ratio` (user), `habit_score` (context) |

## 0. Setup & configuration

In [1]:
import os, sys, json, math, time, gc
from collections import Counter
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

# ── Paths ─────────────────────────────────────────────────────────────────────
# DATA_ROOT must contain MINDsmall_train/ and MINDsmall_dev/ (each with behaviors.tsv, news.tsv, entity_embedding.vec).
DATA_ROOT = "/content/drive/MyDrive/Y4S1/BT4222/"
OUT_DIR = "/content/drive/MyDrive/Y4S1/BT4222/MLP/"

# ── Feature settings ──────────────────────────────────────────────────────────
SEED = 42
N_NEG = 4                       # negatives per positive in training 
CTR_ALPHA = 20.0                # smoothing strength for historical CTR
LAST_N = 5                      # "last-5 clicks" short-term interest window
USER_POOLING = 'mean'           # 'mean' or 'position' (linearly up-weights recent clicks)
SPARSE_USER_MAX = 5             # sparse user slice: 1 <= history_length < 5
STATS_USE_VAL_FOR_TEST = True   # test-day popularity uses days 1-6 (all strictly before test). False -> days 1-5 only
TEXT_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
EMB_BATCH = 256

TRAIN_DAYS, VAL_DAY, TEST_DAY = [0, 1, 2, 3, 4], 5, 6
SPLITS = ['train', 'val', 'test']

os.makedirs(OUT_DIR, exist_ok=True)
rng = np.random.default_rng(SEED)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 20)
print('OUT_DIR =', OUT_DIR)


def save_table(df, name):
    """Parquet when pyarrow is available, pickle otherwise. load_table() reads either."""
    path = os.path.join(OUT_DIR, name)
    try:
        df.to_parquet(path + '.parquet', index=False)
        return path + '.parquet'
    except ImportError:
        df.to_pickle(path + '.pkl')
        return path + '.pkl'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OUT_DIR = /content/drive/MyDrive/Y4S1/BT4222/MLP/


## 1. Load MIND-small (train + dev)

- MIND-small `train` covers Nov 9–14 and `dev` covers Nov 15, giving the 7 days needed for a 5-1-1 split.
- `news.tsv` is read with `quoting=3` (no quote parsing) because some titles contain unbalanced quotes.

In [2]:
BEH_COLS = ['orig_impression_id', 'user_id', 'time', 'history', 'impressions']
NEWS_COLS = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities']

def find_dir(root, name):
    """Locate folders under root."""
    if os.path.isdir(os.path.join(root, name)):
        return os.path.join(root, name)
    for dirpath, dirnames, _ in os.walk(root):
        if name in dirnames:
            return os.path.join(dirpath, name)
    raise FileNotFoundError(f'{name} not found under {root}')


beh_parts, news_parts, entity_files = [], [], []
for source, folder in [('train', 'MINDsmall_train'), ('dev', 'MINDsmall_dev')]:
    d = find_dir(DATA_ROOT, folder)
    b = pd.read_csv(os.path.join(d, 'behaviors.tsv'), sep='\t', header=None, names=BEH_COLS)
    b['source'] = source
    beh_parts.append(b)
    news_parts.append(pd.read_csv(os.path.join(d, 'news.tsv'), sep='\t', header=None, names=NEWS_COLS, quoting=3))
    entity_files.append(os.path.join(d, 'entity_embedding.vec'))

beh = pd.concat(beh_parts, ignore_index=True)
news = pd.concat(news_parts, ignore_index=True).drop_duplicates('news_id').reset_index(drop=True)
beh['time'] = pd.to_datetime(beh['time'], format='%m/%d/%Y %I:%M:%S %p')

# MIND impression ids restart at 1 in dev -> build one global id, ordered by time
beh = beh.sort_values(['time', 'source', 'orig_impression_id'], kind='mergesort').reset_index(drop=True)
beh['impression_id'] = np.arange(1, len(beh) + 1)

print(f'Impressions: {len(beh):,} | users: {beh.user_id.nunique():,} | articles: {len(news):,}')
print('Time range:', beh.time.min(), '->', beh.time.max())

Impressions: 230,117 | users: 94,057 | articles: 65,238
Time range: 2019-11-09 00:00:19 -> 2019-11-15 23:58:03


## 2. Temporal 5-1-1 split

- Day index = calendar day of the impression (day 0 = first day). 
- Train = days 0–4, val = day 5, test = day 6.

In [3]:
first_day = beh['time'].dt.normalize().min()
beh['day'] = (beh['time'].dt.normalize() - first_day).dt.days.astype(int)
n_days = beh['day'].nunique()
assert n_days == 7, f'Expected 7 days of logs, found {n_days}: {sorted(beh.day.unique())}'

beh['split'] = np.select([beh.day.isin(TRAIN_DAYS), beh.day.eq(VAL_DAY), beh.day.eq(TEST_DAY)],
                         ['train', 'val', 'test'], default='drop')
assert (beh.split != 'drop').all()

split_summary = (beh.groupby('split')
                 .agg(impressions=('impression_id', 'size'), users=('user_id', 'nunique'),
                      start=('time', 'min'), end=('time', 'max'), days=('day', lambda s: str([int(x) for x in sorted(s.unique())])))
                 .reindex(SPLITS))
split_summary['share'] = (split_summary.impressions / len(beh)).round(3)
print(split_summary)

# strict ordering: every train impression happens before every val impression, etc.
assert beh.loc[beh.split == 'train', 'time'].max() < beh.loc[beh.split == 'val', 'time'].min()
assert beh.loc[beh.split == 'val', 'time'].max() < beh.loc[beh.split == 'test', 'time'].min()

       impressions  users               start                 end             days  share
split                                                                                    
train       126695  46012 2019-11-09 00:00:19 2019-11-13 23:59:21  [0, 1, 2, 3, 4]  0.551
val          30270  20179 2019-11-14 00:00:00 2019-11-14 23:59:13              [5]  0.132
test         73152  50000 2019-11-15 00:00:01 2019-11-15 23:58:03              [6]  0.318


## 3. Parse impressions, global article index & training-period vocabularies
* `news_idx` 0 is reserved for padding; real articles are 1..N.
* Category / subcategory vocabularies are built **only from articles seen in the training period** (training candidates + training histories). Index 0 = PAD, 1 = UNK. Categories first seen in val/test map to UNK, which the model trains through id-dropout.
* Raw MIND `category` (18) and `subcategory` (~270) are used, and the **same** category vocabulary is used on the user side and article side (shared table).

In [4]:
news['abstract'] = news['abstract'].fillna('')
news['title'] = news['title'].fillna('')
news_id2idx = {nid: i + 1 for i, nid in enumerate(news['news_id'])}
N_ART = len(news) + 1                                   # + PAD row
news['news_idx'] = np.arange(1, N_ART)


def parse_history(h):
    if not isinstance(h, str) or not h.strip():
        return np.zeros(0, dtype=np.int32)
    return np.array([news_id2idx[x] for x in h.split() if x in news_id2idx], dtype=np.int32)


def parse_impressions(s):
    ids, labels = [], []
    for item in s.split():
        nid, lab = item.rsplit('-', 1)
        ids.append(news_id2idx[nid])
        labels.append(int(lab))
    return np.array(ids, dtype=np.int32), np.array(labels, dtype=np.int8)


beh['history'] = beh['history'].fillna('')
parsed = beh['impressions'].map(parse_impressions)
beh['cand_idx'] = parsed.map(lambda t: t[0])
beh['cand_lab'] = parsed.map(lambda t: t[1])
beh['n_pos'] = beh['cand_lab'].map(lambda a: int(a.sum()))
beh['n_cand'] = beh['cand_idx'].map(len)
beh = beh.drop(columns=['impressions'])

# articles seen during the training period (as candidate or in a training-impression history)
tr = beh[beh.split == 'train']
train_hist_idx = np.unique(np.concatenate([parse_history(h) for h in tr['history'].unique()] + [np.zeros(0, np.int32)]))
train_cand_idx = np.unique(np.concatenate(tr['cand_idx'].tolist()))
in_train = np.zeros(N_ART, dtype=bool)
in_train[train_hist_idx] = True
in_train[train_cand_idx] = True
news['in_train'] = in_train[news['news_idx'].values]


def build_vocab(values):
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for v in sorted(set(values)):
        vocab[v] = len(vocab)
    return vocab


cat_vocab = build_vocab(news.loc[news.in_train, 'category'])
subcat_vocab = build_vocab(news.loc[news.in_train, 'subcategory'])
news['category_idx'] = news['category'].map(cat_vocab).fillna(1).astype(int)
news['subcategory_idx'] = news['subcategory'].map(subcat_vocab).fillna(1).astype(int)

art_cat = np.zeros(N_ART, dtype=np.int32);    art_cat[news.news_idx.values] = news.category_idx.values
art_subcat = np.zeros(N_ART, dtype=np.int32); art_subcat[news.news_idx.values] = news.subcategory_idx.values
N_CAT, N_SUBCAT = len(cat_vocab), len(subcat_vocab)

print(f'Articles: {N_ART-1:,} ({news.in_train.sum():,} seen in training period)')
print(f'Category vocab: {N_CAT} (incl. PAD/UNK) | subcategory vocab: {N_SUBCAT} (incl. PAD/UNK)')
print('Articles mapped to UNK category:', int((news.category_idx == 1).sum()),
      '| UNK subcategory:', int((news.subcategory_idx == 1).sum()))
print('Impressions with 0 clicks:', int((beh.n_pos == 0).sum()))

Articles: 65,238 (47,538 seen in training period)
Category vocab: 19 (incl. PAD/UNK) | subcategory vocab: 263 (incl. PAD/UNK)
Articles mapped to UNK category: 1 | UNK subcategory: 9
Impressions with 0 clicks: 0


## 4. Article group
### 4.1 `article_content_emb` — MiniLM 384-d on *title + abstract*

- all-MiniLM-L6-v2 encoder on one embedding of the concatenated text (title + abstract). Missing abstracts are filled with the title. 
- Vectors are L2-normalised so dot product = cosine. 
- Metadata only, no labels → no leakage.

In [5]:
EMB_PATH = os.path.join(OUT_DIR, 'article_content_emb.npy')
EMB_IDS_PATH = os.path.join(OUT_DIR, 'article_content_emb_ids.json')

texts = (news['title'] + '. ' + np.where(news['abstract'].str.strip() == '', news['title'], news['abstract'])).tolist()

cached = False
if os.path.exists(EMB_PATH) and os.path.exists(EMB_IDS_PATH):
    with open(EMB_IDS_PATH) as f:
        cached = json.load(f) == news['news_id'].tolist()
if cached:
    content_emb = np.load(EMB_PATH)
    print('Loaded cached article_content_emb', content_emb.shape)
else:
    import torch
    from sentence_transformers import SentenceTransformer
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    st = SentenceTransformer(TEXT_MODEL, device=device)
    enc = st.encode(texts, batch_size=EMB_BATCH, show_progress_bar=True,
                    convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    content_emb = np.vstack([np.zeros((1, enc.shape[1]), np.float32), enc])   # row 0 = PAD
    np.save(EMB_PATH, content_emb)
    with open(EMB_IDS_PATH, 'w') as f:
        json.dump(news['news_id'].tolist(), f)
    del st
    print('Encoded article_content_emb', content_emb.shape)

assert content_emb.shape[0] == N_ART
D_TEXT = content_emb.shape[1]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/255 [00:00<?, ?it/s]

Encoded article_content_emb (65239, 384)


### 4.2 `kg_entity_emb` (TransE 100-d) + `has_kg_entity`

- Mean of the TransE vectors of all Wikidata entities in the title and abstract, L2-normalised. 
- Articles without a linked entity get `has_kg_entity = 0`; their vector is filled with the training mean in Section 8.

In [6]:
entity_vecs = {}
for path in entity_files:
    if not os.path.exists(path):
        continue
    with open(path, encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            if len(parts) > 2 and parts[0] not in entity_vecs:
                entity_vecs[parts[0]] = np.asarray(parts[1:], dtype=np.float32)
D_KG = len(next(iter(entity_vecs.values())))
print(f'TransE vectors loaded: {len(entity_vecs):,} ({D_KG}-d)')


def entity_ids(s):
    if not isinstance(s, str) or not s.strip():
        return []
    try:
        return [e['WikidataId'] for e in json.loads(s) if isinstance(e, dict) and 'WikidataId' in e]
    except (json.JSONDecodeError, TypeError):
        return []


kg_raw = np.full((N_ART, D_KG), np.nan, dtype=np.float32)
for idx, t_ent, a_ent in zip(news.news_idx, news.title_entities, news.abstract_entities):
    vecs = [entity_vecs[e] for e in set(entity_ids(t_ent) + entity_ids(a_ent)) if e in entity_vecs]
    if vecs:
        v = np.mean(vecs, axis=0)
        n = np.linalg.norm(v)
        kg_raw[idx] = v / n if n > 0 else v
has_kg = ~np.isnan(kg_raw[:, 0])
has_kg[0] = False
news['has_kg_entity'] = has_kg[news.news_idx.values].astype(int)
print(f'Articles with >=1 linked KG entity: {has_kg[1:].mean():.2%}')

TransE vectors loaded: 31,451 (100-d)
Articles with >=1 linked KG entity: 87.03%


## 5. Popularity group (leak-free, lagged by day)
For an impression on day *d*, using only impressions on days **< d** (full slates, not the sampled ones):
* `hist_ctr` = (clicks + α·prior) / (exposures + α), with α = `CTR_ALPHA` and prior = global CTR of the training period. Unseen articles get the prior, i.e. a low-confidence signal.
* `log_exposure` = log1p(exposures)
* `is_cold_article` = 1 if the article has **no clicks** before day *d* (contract cold-article definition).

With `STATS_USE_VAL_FOR_TEST = True`, test (day 7) uses days 1–6, which is still strictly before the test day. Set it to `False` to freeze the stats at the end of training.

In [7]:
day_arr = np.repeat(beh['day'].values, beh['n_cand'].values)
cand_flat = np.concatenate(beh['cand_idx'].tolist())
lab_flat = np.concatenate(beh['cand_lab'].tolist()).astype(np.int64)

exp_by_day = np.zeros((7, N_ART), dtype=np.int64)
clk_by_day = np.zeros((7, N_ART), dtype=np.int64)
np.add.at(exp_by_day, (day_arr, cand_flat), 1)
np.add.at(clk_by_day, (day_arr, cand_flat), lab_flat)

# cum_*[d] = totals over days strictly before d
cum_exp = np.vstack([np.zeros((1, N_ART), np.int64), np.cumsum(exp_by_day, 0)[:-1]])
cum_clk = np.vstack([np.zeros((1, N_ART), np.int64), np.cumsum(clk_by_day, 0)[:-1]])

CTR_PRIOR = clk_by_day[TRAIN_DAYS].sum() / exp_by_day[TRAIN_DAYS].sum()
LAST_STATS_DAY = VAL_DAY if STATS_USE_VAL_FOR_TEST else TRAIN_DAYS[-1]


def stats_day(d):
    """Index into cum_* for an impression on day d (never includes day d itself)."""
    return np.minimum(d, LAST_STATS_DAY + 1)


del day_arr, cand_flat, lab_flat
print(f'Training-period CTR prior = {CTR_PRIOR:.4f}')
for d in range(7):
    sd = stats_day(d)
    print(f'day {d}: stats from days < {sd} | articles with >=1 prior click: {(cum_clk[sd] > 0).sum():,}')

Training-period CTR prior = 0.0410
day 0: stats from days < 0 | articles with >=1 prior click: 0
day 1: stats from days < 1 | articles with >=1 prior click: 1,378
day 2: stats from days < 2 | articles with >=1 prior click: 2,283
day 3: stats from days < 3 | articles with >=1 prior click: 3,736
day 4: stats from days < 4 | articles with >=1 prior click: 5,046
day 5: stats from days < 5 | articles with >=1 prior click: 6,398
day 6: stats from days < 6 | articles with >=1 prior click: 7,713


## 6. User profiles
MIND's `history` field is the user's click history **before** the impression, so it is leak-free by construction. A profile is computed once per distinct history:
* `user_history_emb`: pooled `article_content_emb` of clicked articles (mean, or position-weighted), L2-normalised
* `history_length`, `log_history_length`, `has_history`
* `reading_entropy`: Shannon entropy of the history's category distribution
* `user_category_affinity`: share of history clicks per category, aligned with the **shared category vocabulary** (the model multiplies it by the article category embedding table)
* `diversity_ratio` (optional): unique categories / history length

Profile 0 is the empty history (cold user). Its values are NaN here and filled with training means in Section 8.

In [8]:
beh['profile_key'] = beh['history'].str.strip()
uniq_hist = pd.Index(beh['profile_key'].unique())
if '' in uniq_hist:
    uniq_hist = pd.Index([''] + [h for h in uniq_hist if h != ''])
else:
    uniq_hist = pd.Index([''] + list(uniq_hist))
beh['profile_idx'] = uniq_hist.get_indexer(beh['profile_key'])
P = len(uniq_hist)

hist_lists = [parse_history(h) for h in uniq_hist]
hist_len = np.array([len(h) for h in hist_lists], dtype=np.int32)

user_emb = np.full((P, D_TEXT), np.nan, dtype=np.float32)
user_last = np.full((P, D_TEXT), np.nan, dtype=np.float32)
user_kg = np.full((P, D_KG), np.nan, dtype=np.float32)
user_aff = np.full((P, N_CAT), np.nan, dtype=np.float32)
entropy = np.full(P, np.nan, dtype=np.float32)
diversity = np.full(P, np.nan, dtype=np.float32)
top_cat = np.zeros(P, dtype=np.int32)          # 0 = undefined (cold)


def l2n(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else v


t0 = time.time()
for p, h in enumerate(hist_lists):
    if len(h) == 0:
        continue
    E = content_emb[h]
    if USER_POOLING == 'position':
        w = np.arange(1, len(h) + 1, dtype=np.float32)
        user_emb[p] = l2n((w[:, None] * E).sum(0) / w.sum())
    else:
        user_emb[p] = l2n(E.mean(0))
    user_last[p] = l2n(E[-LAST_N:].mean(0))               # MIND histories are in click order (latest last)
    kh = h[has_kg[h]]
    if len(kh):
        user_kg[p] = l2n(kg_raw[kh].mean(0))
    counts = np.bincount(art_cat[h], minlength=N_CAT).astype(np.float32)
    share = counts / counts.sum()
    user_aff[p] = share
    nz = share[share > 0]
    entropy[p] = float(-(nz * np.log(nz)).sum())
    diversity[p] = len(nz) / len(h)
    top_cat[p] = int(np.argmax(counts))
print(f'{P:,} profiles built in {time.time()-t0:.1f}s | cold-profile share of impressions: '
      f'{(beh.profile_idx == 0).mean():.2%}')

91,359 profiles built in 12.4s | cold-profile share of impressions: 2.37%


## 7. Candidate tables & the negative-sampling policy
**Training**: Each clicked article becomes one *group* = [positive, up to K=4 negatives]. Negatives are drawn from the same impression's non-clicked articles **without replacement** (seed 42). If an impression has fewer than 4 negatives, the group keeps the ones it has; the model masks the empty slots, so no negative is ever duplicated. Impressions without any negative are dropped from training.

**Val/Test**: One group per impression with the **full slate** of candidates (original order), so evaluation ranks exactly what the user saw. Impressions without a click are dropped (no metric is defined).

In [9]:
def build_groups(df, split):
    g_imp, g_slots = [], []
    if split == 'train':
        for imp_id, c, l in zip(df.impression_id.values, df.cand_idx.values, df.cand_lab.values):
            pos, neg = c[l == 1], c[l == 0]
            if len(neg) == 0:
                continue
            for pnews in pos:
                k = min(N_NEG, len(neg))
                g_imp.append(imp_id)
                g_slots.append((np.concatenate([[pnews], rng.choice(neg, size=k, replace=False)]),
                                np.concatenate([[1], np.zeros(k, np.int8)])))
    else:
        for imp_id, c, l in zip(df.impression_id.values, df.cand_idx.values, df.cand_lab.values):
            if l.sum() == 0:
                continue
            g_imp.append(imp_id)
            g_slots.append((c, l))
    n = np.array([len(s[0]) for s in g_slots], dtype=np.int32)
    groups = pd.DataFrame({'group_id': np.arange(len(g_imp)), 'impression_id': g_imp,
                           'cand_start': np.concatenate([[0], np.cumsum(n)[:-1]]), 'n_cand': n})
    cands = pd.DataFrame({
        'group_id': np.repeat(groups.group_id.values, n),
        'impression_id': np.repeat(groups.impression_id.values, n),
        'slot': np.concatenate([np.arange(k) for k in n]).astype(np.int16),
        'news_idx': np.concatenate([s[0] for s in g_slots]).astype(np.int32),
        'label': np.concatenate([s[1] for s in g_slots]).astype(np.int8),
    })
    return groups, cands


tables = {}
for split in SPLITS:
    df = beh[beh.split == split]
    groups, cands = build_groups(df, split)
    tables[split] = (groups, cands)
    print(f'{split:5s}: {len(groups):>8,} groups | {len(cands):>10,} candidate rows | '
          f'mean slate {groups.n_cand.mean():.1f} | positives {int(cands.label.sum()):,}')

g_tr = tables['train'][0]
print(f'Train groups with < {N_NEG} negatives (masked, not padded with repeats): '
      f'{(g_tr.n_cand < N_NEG + 1).sum():,} ({(g_tr.n_cand < N_NEG + 1).mean():.2%})')

train:  189,519 groups |    906,625 candidate rows | mean slate 4.8 | positives 189,519
val  :   30,270 groups |  1,222,429 candidate rows | mean slate 40.4 | positives 46,825
test :   73,152 groups |  2,740,998 candidate rows | mean slate 37.5 | positives 111,383
Train groups with < 4 negatives (masked, not padded with repeats): 19,075 (10.06%)


### 7.1 Candidate-level features (popularity + user × article)
* `cos_user_hist` = cos(`user_history_emb`, `article_content_emb`)
* `max_cos_hist` = highest cosine between the candidate and **any single** clicked article
* `cos_last5` = cosine with the mean of the 5 most recent clicks (short-term interest)
* `cat_affinity` = share of the user's clicks in the candidate's category
* `category_match` = 1 if the candidate's category is the user's top category
* `kg_match` = cos(user TransE profile, article TransE vector); NaN if either side has no entity

All user × article features are NaN for cold users (filled with training means in Section 8; `has_history` tells the model).

In [10]:
imp_info = beh.set_index('impression_id')[['day', 'profile_idx']]


def rowwise_dot(A, ia, B, ib, chunk=100_000):
    out = np.empty(len(ia), dtype=np.float32)
    for s in range(0, len(ia), chunk):
        out[s:s+chunk] = np.einsum('ij,ij->i', A[ia[s:s+chunk]], B[ib[s:s+chunk]])
    return out


def add_candidate_features(cands):
    info = imp_info.loc[cands.impression_id.values]
    day = info['day'].values
    prof = info['profile_idx'].values
    nidx = cands['news_idx'].values
    cold_user = hist_len[prof] == 0

    # popularity (lagged)
    sd = stats_day(day)
    clk, ex = cum_clk[sd, nidx], cum_exp[sd, nidx]
    cands['hist_ctr'] = ((clk + CTR_ALPHA * CTR_PRIOR) / (ex + CTR_ALPHA)).astype(np.float32)
    cands['log_exposure'] = np.log1p(ex).astype(np.float32)
    cands['is_cold_article'] = (clk == 0).astype(np.int8)

    # user x article
    cands['cos_user_hist'] = rowwise_dot(user_emb, prof, content_emb, nidx)
    cands['cos_last5'] = rowwise_dot(user_last, prof, content_emb, nidx)
    cat = art_cat[nidx]
    cands['cat_affinity'] = user_aff[prof, cat]
    cm = (cat == top_cat[prof]).astype(np.float32)
    cm[cold_user] = np.nan
    cands['category_match'] = cm
    kgm = rowwise_dot(np.nan_to_num(user_kg), prof, np.nan_to_num(kg_raw), nidx)
    kgm[~(has_kg[nidx] & ~np.isnan(user_kg[prof, 0]))] = np.nan
    cands['kg_match'] = kgm

    # max cosine over the full history: one matrix product per (profile, group)
    mx = np.full(len(cands), np.nan, dtype=np.float32)
    order = np.argsort(prof, kind='stable')
    bounds = np.flatnonzero(np.diff(prof[order])) + 1
    for rows in np.split(order, bounds):
        h = hist_lists[prof[rows[0]]]
        if len(h):
            mx[rows] = (content_emb[nidx[rows]] @ content_emb[h].T).max(1)
    cands['max_cos_hist'] = mx
    return cands


for split in SPLITS:
    t0 = time.time()
    groups, cands = tables[split]
    tables[split] = (groups, add_candidate_features(cands))
    print(f'{split}: candidate features in {time.time()-t0:.1f}s')

train: candidate features in 8.9s
val: candidate features in 6.2s
test: candidate features in 13.9s


### 7.2 Impression-level context
* `hour_sin`, `hour_cos` of the impression hour
* `habit_score` (optional): cos(2π·Δt/24), Δt = hours since the user's previous impression (any earlier impression, always in the past). NaN for a user's first visit.

In [11]:
beh = beh.sort_values(['user_id', 'time', 'impression_id'], kind='mergesort')
gap_h = (beh['time'] - beh.groupby('user_id')['time'].shift(1)).dt.total_seconds() / 3600.0
beh['habit_score'] = np.cos(2 * np.pi * gap_h / 24.0).astype(np.float32)
beh = beh.sort_values('impression_id').reset_index(drop=True)
hour = beh['time'].dt.hour + beh['time'].dt.minute / 60.0
beh['hour_sin'] = np.sin(2 * np.pi * hour / 24).astype(np.float32)
beh['hour_cos'] = np.cos(2 * np.pi * hour / 24).astype(np.float32)

## 8. Missing values imputed with training-set mean 
Means are computed on the **training split only** and applied to all three splits:
* Candidate Features: Mean over training candidate rows
* Profile Features (`user_history_emb`, `reading_entropy`, `diversity_ratio`, `user_category_affinity`): Mean over training impressions of users with history
* `kg_entity_emb`: Mean over training-period articles that have entities
* `habit_score`: Mean over training impressions

The cold-start flags (`has_history`, `has_kg_entity`, `is_cold_article`) stay, so the model can tell an imputed value from a real one.

In [12]:
CAND_NUM = ['hist_ctr', 'log_exposure', 'is_cold_article',
            'category_match', 'cos_user_hist', 'max_cos_hist', 'cos_last5', 'cat_affinity', 'kg_match']
train_c = tables['train'][1]
impute_means = {c: float(np.nanmean(train_c[c].values.astype(np.float64))) for c in CAND_NUM}
missing_report = {}
for split in SPLITS:
    g, c = tables[split]
    missing_report[split] = {k: int(c[k].isna().sum()) for k in CAND_NUM}
    for k in CAND_NUM:
        c[k] = c[k].fillna(impute_means[k]).astype(np.float32)
print('NaNs filled per split:')
print(pd.DataFrame(missing_report))

# profile-level
tr_imp = beh[beh.split == 'train']
tr_prof = tr_imp['profile_idx'].values
tr_prof_warm = tr_prof[hist_len[tr_prof] > 0]
user_emb_mean = user_emb[tr_prof_warm].mean(0)
user_aff_mean = user_aff[tr_prof_warm].mean(0)
entropy_mean = float(entropy[tr_prof_warm].mean())
diversity_mean = float(diversity[tr_prof_warm].mean())
cold_rows = hist_len == 0
user_emb[cold_rows] = user_emb_mean
user_aff[cold_rows] = user_aff_mean
entropy[cold_rows] = entropy_mean
diversity[cold_rows] = diversity_mean

# KG article vectors
kg_train_mask = has_kg & in_train
kg_mean = kg_raw[kg_train_mask].mean(0)
kg_emb = kg_raw.copy()
kg_emb[~has_kg] = kg_mean
kg_emb[0] = 0.0                                       # PAD row

habit_mean = float(np.nanmean(beh.loc[beh.split == 'train', 'habit_score']))
beh['habit_score'] = beh['habit_score'].fillna(habit_mean).astype(np.float32)

impute_means.update({'user_history_emb': 'vector (train mean)', 'user_category_affinity': 'vector (train mean)',
                     'reading_entropy': entropy_mean, 'diversity_ratio': diversity_mean,
                     'kg_entity_emb': 'vector (train mean)', 'habit_score': habit_mean})
for k, v in impute_means.items():
    print(f'  {k:24s} {v if isinstance(v, str) else round(v, 5)}')
assert not np.isnan(user_emb).any() and not np.isnan(kg_emb).any() and not np.isnan(user_aff).any()

NaNs filled per split:
                  train     val    test
hist_ctr              0       0       0
log_exposure          0       0       0
is_cold_article       0       0       0
category_match    17274   30575   82907
cos_user_hist     17274   30575   82907
max_cos_hist      17274   30575   82907
cos_last5         17274   30575   82907
cat_affinity      17274   30575   82907
kg_match         188540  289390  646793
  hist_ctr                 0.04417
  log_exposure             2.58631
  is_cold_article          0.65208
  category_match           0.24592
  cos_user_hist            0.18081
  max_cos_hist             0.26353
  cos_last5                0.12936
  cat_affinity             0.1744
  kg_match                 0.45148
  user_history_emb         vector (train mean)
  user_category_affinity   vector (train mean)
  reading_entropy          1.52033
  diversity_ratio          0.40804
  kg_entity_emb            vector (train mean)
  habit_score              0.45055


## 9. Export (identical format for every split)
```
OUT_DIR/
  articles.parquet            news_idx, news_id, category_idx, subcategory_idx, has_kg_entity, in_train
  article_content_emb.npy     [N_ART, 384]   row 0 = PAD
  kg_entity_emb.npy           [N_ART, 100]   imputed
  user_profiles.parquet       profile_idx, history_length, log_history_length, has_history, reading_entropy, diversity_ratio
  user_history_emb.npy        [P, 384]
  user_category_affinity.npy  [P, N_CAT]
  {split}_impressions.parquet impression_id, orig_impression_id, source, user_id, time, day, profile_idx,
                              history_length, hour_sin, hour_cos, habit_score, n_pos, n_cand
  {split}_groups.parquet      group_id, impression_id, cand_start, n_cand
  {split}_candidates.parquet  group_id, impression_id, slot, news_idx, label, <9 candidate features>
  feature_meta.json           feature lists, vocab sizes, imputation means, split & cold-start definitions
```

In [13]:
articles = news[['news_idx', 'news_id', 'category', 'subcategory', 'category_idx', 'subcategory_idx',
                 'has_kg_entity', 'in_train']].copy()
save_table(articles, 'articles')
np.save(os.path.join(OUT_DIR, 'kg_entity_emb.npy'), kg_emb.astype(np.float32))

profiles = pd.DataFrame({
    'profile_idx': np.arange(P), 'history_length': hist_len,
    'log_history_length': np.log1p(hist_len).astype(np.float32),
    'has_history': (hist_len > 0).astype(np.int8),
    'reading_entropy': entropy, 'diversity_ratio': diversity})
save_table(profiles, 'user_profiles')
np.save(os.path.join(OUT_DIR, 'user_history_emb.npy'), user_emb.astype(np.float32))
np.save(os.path.join(OUT_DIR, 'user_category_affinity.npy'), user_aff.astype(np.float32))

IMP_COLS = ['impression_id', 'orig_impression_id', 'source', 'user_id', 'time', 'day', 'profile_idx',
            'hour_sin', 'hour_cos', 'habit_score', 'n_pos', 'n_cand']
for split in SPLITS:
    g, c = tables[split]
    imp = beh.loc[beh.impression_id.isin(g.impression_id.unique()), IMP_COLS].copy()
    imp['history_length'] = hist_len[imp.profile_idx.values]
    save_table(imp, f'{split}_impressions')
    save_table(g, f'{split}_groups')
    save_table(c, f'{split}_candidates')

meta = {
    'split': {'scheme': '5-1-1 by calendar day', 'first_day': str(first_day.date()),
              'train_days': TRAIN_DAYS, 'val_day': VAL_DAY, 'test_day': TEST_DAY,
              'summary': {s: {'impressions': int(split_summary.loc[s, 'impressions']),
                              'start': str(split_summary.loc[s, 'start']), 'end': str(split_summary.loc[s, 'end'])}
                          for s in SPLITS}},
    'negative_sampling': {'train': f'per positive, up to {N_NEG} negatives from same impression, no replacement, '
                                   f'seed {SEED}; short groups masked (no repeats); impressions without negatives dropped',
                          'val_test': 'full impression slate; impressions without clicks dropped'},
    'cold_start': {'sparse_user_max': SPARSE_USER_MAX,
                   'cold_user': 'history_length == 0',
                   'sparse_user': f'1 <= history_length < {SPARSE_USER_MAX}',
                   'cold_article': 'no clicks in any impression before the impression day (is_cold_article == 1)'},
    'leakage': {'popularity': 'lagged by day (days < impression day)', 'ctr_prior': CTR_PRIOR, 'ctr_alpha': CTR_ALPHA,
                'stats_use_val_for_test': STATS_USE_VAL_FOR_TEST,
                'vocab_and_means': 'training period only'},
    'excluded_features': ['slate_length', 'article_recency', 'session_dow'],
    'features': {
        'article': {'dense': ['article_content_emb', 'kg_entity_emb'], 'flags': ['has_kg_entity'],
                    'categorical': ['category_idx', 'subcategory_idx']},
        'popularity': ['hist_ctr', 'log_exposure', 'is_cold_article'],
        'user': {'dense': ['user_history_emb'], 'scalar': ['log_history_length', 'has_history', 'reading_entropy'],
                 'category_profile': 'user_category_affinity'},
        'user_x_article': ['category_match', 'cos_user_hist', 'max_cos_hist', 'cos_last5', 'cat_affinity', 'kg_match'],
        'context': ['hour_sin', 'hour_cos'],
        'optional': {'user': ['diversity_ratio'], 'context': ['habit_score']},
    },
    'dims': {'text': int(D_TEXT), 'kg': int(D_KG), 'n_cat': int(N_CAT), 'n_subcat': int(N_SUBCAT),
             'n_articles': int(N_ART), 'n_profiles': int(P)},
    'imputation_means': impute_means,
    'config': {'user_pooling': USER_POOLING, 'last_n': LAST_N, 'text_model': TEXT_MODEL},
}
with open(os.path.join(OUT_DIR, 'feature_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2, default=str)
print('Saved to', OUT_DIR)
print(sorted(os.listdir(OUT_DIR)))

Saved to /content/drive/MyDrive/Y4S1/BT4222/MLP/
['article_content_emb.npy', 'article_content_emb_ids.json', 'articles.parquet', 'feature_meta.json', 'kg_entity_emb.npy', 'test_candidates.parquet', 'test_groups.parquet', 'test_impressions.parquet', 'train_candidates.parquet', 'train_groups.parquet', 'train_impressions.parquet', 'user_category_affinity.npy', 'user_history_emb.npy', 'user_profiles.parquet', 'val_candidates.parquet', 'val_groups.parquet', 'val_impressions.parquet']


## 10. Leakage & sanity checks

In [14]:
# (a) Popularity: recount exposures/clicks for sampled test rows straight from raw impressions on earlier days
day_flat = np.repeat(beh['day'].values, beh['n_cand'].values)
cand_flat = np.concatenate(beh['cand_idx'].tolist())
lab_flat = np.concatenate(beh['cand_lab'].tolist())
c = tables['test'][1].sample(min(25, len(tables['test'][1])), random_state=SEED)
for _, r in c.iterrows():
    d = int(imp_info.loc[r.impression_id, 'day'])
    m = (day_flat < stats_day(d)) & (cand_flat == r.news_idx)
    assert abs(np.log1p(m.sum()) - r.log_exposure) < 1e-4, 'exposure count leak/mismatch'
    assert int(lab_flat[m].sum() == 0) == r.is_cold_article, 'cold-article flag mismatch'
del day_flat, cand_flat, lab_flat
print('(a) lagged popularity matches a raw recount (days strictly before the impression day)')

# (b) No test-day or val-day clicks inside training statistics
for d in range(7):
    assert stats_day(d) <= d          # cum_*[stats_day(d)] covers days < stats_day(d) <= d
print('(b) every split only uses popularity from days strictly before its own day')

# (c) Vocab & imputation means used only training-period data
assert set(cat_vocab) - {'<PAD>', '<UNK>'} <= set(news.loc[news.in_train, 'category'])
print('(c) category vocabulary built from training-period articles only')

# (d) Distribution overview
summary = []
for split in SPLITS:
    g, c = tables[split]
    imp = beh[beh.impression_id.isin(g.impression_id)]
    hl = hist_len[imp.profile_idx.values]
    summary.append({'split': split, 'groups': len(g), 'rows': len(c),
                    'cold_user_imps': f'{(hl == 0).mean():.2%}',
                    'sparse_user_imps': f'{((hl > 0) & (hl < SPARSE_USER_MAX)).mean():.2%}',
                    'cold_article_rows': f'{c.is_cold_article.mean():.2%}',
                    'cold_article_positives': f'{c.loc[c.label == 1, "is_cold_article"].mean():.2%}'})
print(pd.DataFrame(summary).to_string(index=False))
print('\nCandidate feature summary (train):')
print(tables['train'][1][CAND_NUM].describe().T[['mean', 'std', 'min', 'max']].round(4))

(a) lagged popularity matches a raw recount (days strictly before the impression day)
(b) every split only uses popularity from days strictly before its own day
(c) category vocabulary built from training-period articles only
split  groups    rows cold_user_imps sparse_user_imps cold_article_rows cold_article_positives
train  189519  906625          1.95%           11.33%            65.21%                 66.35%
  val   30270 1222429          2.55%           10.73%            60.33%                 65.71%
 test   73152 2740998          3.03%           11.06%            54.86%                 51.36%

Candidate feature summary (train):
                   mean     std     min     max
hist_ctr         0.0442  0.0310  0.0010  0.4545
log_exposure     2.5863  2.8845  0.0000  9.8130
is_cold_article  0.6521  0.4757  0.0000  1.0000
category_match   0.2459  0.4272  0.0000  1.0000
cos_user_hist    0.1808  0.1177 -0.2431  1.0000
max_cos_hist     0.2635  0.1278 -0.1982  1.0000
cos_last5        0.129